# ============================================================
# TINYGPT V2
# General English Pretraining → DailyDialog Fine-Tuning
# ============================================================


# ============================================================
# 1. ENVIRONMENT SETUP
# ============================================================


In [ ]:
!pip install -q transformers datasets accelerate tqdm


Importing the libraries.

In [ ]:
import os
import random
import numpy as np
import torch

from tqdm.auto import tqdm

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu130


Checking the gpu , environment test.

In [ ]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    gpu_memory = torch.cuda.get_device_properties(0).total_memory
    print(
        "GPU Memory:",
        round(gpu_memory / (1024 ** 3), 2),
        "GB"
    )
else:
    print("WARNING: CUDA is not available.")

Device: cuda
GPU: Tesla T4
GPU Memory: 14.56 GB


Reproducibility.

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Random seed:", SEED)

Random seed: 42


Mount Google Drive.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


Create project directories


In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/tiny-GPT-v2"

CHECKPOINT_DIR = os.path.join(
    PROJECT_DIR,
    "checkpoints"
)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Checkpoint directory:", CHECKPOINT_DIR)

Project directory: /content/drive/MyDrive/tiny-GPT-v2
Checkpoint directory: /content/drive/MyDrive/tiny-GPT-v2/checkpoints


Optimal gpu optimisation settings.

In [ ]:
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    print("TF32 enabled for CUDA.")

TF32 enabled for CUDA.


# ============================================================
# 2. CONFIGURATION
# ============================================================

Model configuration.

In [ ]:
# Model configuration

VOCAB_SIZE = 50257

CONTEXT_LENGTH = 256

EMBEDDING_DIM = 512

NUM_LAYERS = 8

NUM_HEADS = 8

DROPOUT = 0.1

Training Configuration.

In [ ]:
# Training configuration

BATCH_SIZE = 8

GRADIENT_ACCUMULATION_STEPS = 4

LEARNING_RATE = 3e-4

WEIGHT_DECAY = 0.01

NUM_EPOCHS = 1

Dataset Token Budget.

In [ ]:
# Dataset configuration

DATASET_NAME = "HuggingFaceFW/fineweb"

TRAINING_TOKENS = 50_000_000

VALIDATION_TOKENS = 1_000_000

Training Precision.

In [ ]:
# Mixed precision configuration

USE_BF16 = (
    torch.cuda.is_available()
    and torch.cuda.is_bf16_supported()
)

USE_FP16 = (
    torch.cuda.is_available()
    and not USE_BF16
)

print("BF16:", USE_BF16)
print("FP16:", USE_FP16)

BF16: True
FP16: False


**Print configuration.**

In [ ]:
print("========== TinyGPT v2 Configuration ==========")
print("Vocabulary size:", VOCAB_SIZE)
print("Context length:", CONTEXT_LENGTH)
print("Embedding dimension:", EMBEDDING_DIM)
print("Layers:", NUM_LAYERS)
print("Attention heads:", NUM_HEADS)
print("Batch size:", BATCH_SIZE)
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Learning rate:", LEARNING_RATE)
print("Training tokens:", TRAINING_TOKENS)
print("===============================================")

========== TinyGPT v2 Configuration ==========
Vocabulary size: 50257
Context length: 256
Embedding dimension: 512
Layers: 8
Attention heads: 8
Batch size: 8
Gradient accumulation: 4
Learning rate: 0.0003
Training tokens: 50000000


# ============================================================
# 3. TOKENIZER
# ============================================================

In [ ]:
# ============================================================
# 3. TOKENIZER
# ============================================================

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")

print("Tokenizer:", tokenizer.name_or_path)
print("Vocabulary size:", tokenizer.vocab_size)

test_text = "Hello! This is TinyGPT v2 learning English."

tokens = tokenizer(test_text)

print("Token IDs:", tokens["input_ids"])
print("Number of tokens:", len(tokens["input_ids"]))

decoded = tokenizer.decode(tokens["input_ids"])

print("Decoded text:", decoded)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Tokenizer: gpt2
Vocabulary size: 50257
Token IDs: [15496, 0, 770, 318, 20443, 38, 11571, 410, 17, 4673, 3594, 13]
Number of tokens: 12
Decoded text: Hello! This is TinyGPT v2 learning English.


# ============================================================
# 4. MODEL
# ============================================================

Creating the model.

In [ ]:
# ============================================================
# 4. MODEL
# ============================================================

from transformers import GPT2Config, GPT2LMHeadModel

config = GPT2Config(
    vocab_size=VOCAB_SIZE,
    n_positions=CONTEXT_LENGTH,
    n_ctx=CONTEXT_LENGTH,
    n_embd=EMBEDDING_DIM,
    n_layer=NUM_LAYERS,
    n_head=NUM_HEADS,
    resid_pdrop=DROPOUT,
    embd_pdrop=DROPOUT,
    attn_pdrop=DROPOUT,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id
)

model = GPT2LMHeadModel(config)

model = model.to(device)

Checking parameters.

In [ ]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print(
    "Parameters (millions):",
    round(total_params / 1e6, 2),
    "M"
)

Total parameters: 51082752
Trainable parameters: 51082752
Parameters (millions): 51.08 M


Confirming random initialisation.

In [ ]:
print(model)

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 512)
    (wpe): Embedding(256, 512)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-7): 8 x GPT2Block(
        (ln_1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=1536, nx=512)
          (c_proj): Conv1D(nf=512, nx=512)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=2048, nx=512)
          (c_proj): Conv1D(nf=512, nx=2048)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=512, out_features=50257, bias=False)
)


Quick forward-pass test.

In [ ]:
test_text = "Hello, how are you?"

test_tokens = tokenizer(
    test_text,
    return_tensors="pt"
)

test_input_ids = test_tokens["input_ids"].to(device)

with torch.no_grad():
    outputs = model(
        input_ids=test_input_ids
    )

print("Input shape:", test_input_ids.shape)
print("Logits shape:", outputs.logits.shape)

Input shape: torch.Size([1, 6])
Logits shape: torch.Size([1, 6, 50257])


# ============================================================
# 5. GENERAL ENGLISH DATASET
# ============================================================

Load fineweb in streaming mode.

In [ ]:
from datasets import load_dataset

fineweb = load_dataset(
    DATASET_NAME,
    split="train",
    streaming=True
)

print(fineweb)

README.md:   0%|          | 0.00/44.3k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

IterableDataset({
    features: ['text', 'id', 'dump', 'url', 'date', 'file_path', 'language', 'language_score', 'token_count'],
    num_shards: 27468
})


Inspecting samples,

In [ ]:
sample_iterator = iter(fineweb)

for i in range(3):
    sample = next(sample_iterator)

    print("\n" + "=" * 60)
    print(f"SAMPLE {i + 1}")
    print("=" * 60)
    print(sample["text"][:1000])


SAMPLE 1
How AP reported in all formats from tornado-stricken regionsMarch 8, 2012
When the first serious bout of tornadoes of 2012 blew through middle America in the middle of the night, they touched down in places hours from any AP bureau. Our closest video journalist was Chicago-based Robert Ray, who dropped his plans to travel to Georgia for Super Tuesday, booked several flights to the cities closest to the strikes and headed for the airport. He’d decide once there which flight to take.
He never got on board a plane. Instead, he ended up driving toward Harrisburg, Ill., where initial reports suggested a town was destroyed. That decision turned out to be a lucky break for the AP. Twice.
Ray was among the first journalists to arrive and he confirmed those reports -- in all formats. He shot powerful video, put victims on the phone with AP Radio and played back sound to an editor who transcribed the interviews and put the material on text wires. He then walked around the devastation w

Quick tokenizer test.

In [ ]:
text = sample["text"]

tokens = tokenizer(
    text,
    add_special_tokens=False
)["input_ids"]

print("Characters:", len(text))
print("Tokens:", len(tokens))
print("\nFirst 50 token IDs:")
print(tokens[:50])

Characters: 456
Tokens: 125

First 50 token IDs:
[9914, 22892, 1114, 27443, 0, 198, 3844, 318, 534, 2863, 284, 1037, 0, 362, 614, 1468, 27443, 40782, 468, 15523, 0, 27443, 460, 447, 247, 83, 910, 564, 250, 8199, 1434, 39806, 6086, 447, 251, 475, 673, 4206, 703, 340, 5300, 13, 921, 460, 1037, 3228, 198, 32, 1879, 22892]


# ============================================================
# 6. DATA PREPROCESSING
# ============================================================

Text cleaning + tokenization

In [ ]:
# ============================================================
# 6. DATA PREPROCESSING
# ============================================================

def tokenize_text(text):
    """
    Clean and tokenize one FineWeb document.
    """
    text = text.strip()

    if not text:
        return []

    token_ids = tokenizer(
        text,
        add_special_tokens=False
    )["input_ids"]

    return token_ids

Create continuos token streams.

In [ ]:
def token_stream(dataset_stream):
    """
    Convert streamed documents into one continuous
    stream of token IDs.
    """

    for sample in dataset_stream:

        token_ids = tokenize_text(sample["text"])

        if not token_ids:
            continue

        for token_id in token_ids:
            yield token_id

        # Mark end of document
        yield tokenizer.eos_token_id

Create training chunks.

In [ ]:
def chunk_token_stream(token_stream, context_length):
    """
    Convert continuous token stream into fixed-length
    input/label pairs.
    """

    buffer = []

    for token_id in token_stream:

        buffer.append(token_id)

        while len(buffer) >= context_length + 1:

            chunk = buffer[:context_length + 1]

            input_ids = chunk[:-1]
            labels = chunk[1:]

            yield (
                torch.tensor(input_ids, dtype=torch.long),
                torch.tensor(labels, dtype=torch.long)
            )

            # Keep one-token overlap for next prediction window
            buffer = buffer[context_length:]

Testing the traning pipe line.



In [ ]:
from datasets import load_dataset

test_fineweb = load_dataset(
    DATASET_NAME,
    split="train",
    streaming=True
)

tokens = token_stream(test_fineweb)

chunks = chunk_token_stream(
    tokens,
    CONTEXT_LENGTH
)

input_ids, labels = next(chunks)

print("Input shape:", input_ids.shape)
print("Labels shape:", labels.shape)

print("First 20 input IDs:")
print(input_ids[:20])

print("First 20 labels:")
print(labels[:20])

print("\nDecoded input:")
print(tokenizer.decode(input_ids.tolist()))

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Input shape: torch.Size([256])
Labels shape: torch.Size([256])
First 20 input IDs:
tensor([ 2437,  3486,  2098,   287,   477, 17519,   422, 33718,    12,  2536,
         5973,  7652, 16192,   807,    11,  2321,   198,  2215,   262,   717])
First 20 labels:
tensor([ 3486,  2098,   287,   477, 17519,   422, 33718,    12,  2536,  5973,
         7652, 16192,   807,    11,  2321,   198,  2215,   262,   717,  2726])

Decoded input:
How AP reported in all formats from tornado-stricken regionsMarch 8, 2012
When the first serious bout of tornadoes of 2012 blew through middle America in the middle of the night, they touched down in places hours from any AP bureau. Our closest video journalist was Chicago-based Robert Ray, who dropped his plans to travel to Georgia for Super Tuesday, booked several flights to the cities closest to the strikes and headed for the airport. He’d decide once there which flight to take.
He never got on board a plane. Instead, he ended up driving toward Harrisburg, Ill.

# ============================================================
# 7. PYTORCH DATASET + DATALOADER
# ============================================================

Streaming dataset.

In [ ]:
from torch.utils.data import IterableDataset, DataLoader
import hashlib


class FineWebDataset(IterableDataset):

    def __init__(
        self,
        token_budget,
        context_length,
        validation=False
    ):
        self.token_budget = token_budget
        self.context_length = context_length
        self.validation = validation

        self.num_examples = (
            token_budget // context_length
        )
    def __len__(self):
      return self.num_examples

    def _is_validation(self, text):
        # Deterministic 5% validation split
        value = int(
            hashlib.md5(
                text.encode("utf-8")
            ).hexdigest(),
            16
        )

        return (value % 100) < 5

    def __iter__(self):

        dataset_stream = load_dataset(
            DATASET_NAME,
            split="train",
            streaming=True
        )

        examples_produced = 0
        buffer = []

        for sample in dataset_stream:

            text = sample["text"].strip()

            if not text:
                continue

            # Keep validation and training documents separate
            if self._is_validation(text) != self.validation:
                continue

            token_ids = tokenizer(
                text,
                add_special_tokens=False
            )["input_ids"]

            if not token_ids:
                continue

            buffer.extend(token_ids)

            # EOS marks document boundary
            buffer.append(tokenizer.eos_token_id)

            while len(buffer) >= self.context_length + 1:

                chunk = buffer[
                    :self.context_length + 1
                ]

                input_ids = chunk[:-1]
                labels = chunk[1:]

                yield (
                    torch.tensor(
                        input_ids,
                        dtype=torch.long
                    ),
                    torch.tensor(
                        labels,
                        dtype=torch.long
                    )
                )

                examples_produced += 1

                if examples_produced >= self.num_examples:
                    return

                buffer = buffer[
                    self.context_length:
                ]

Create train/validation dataset.

In [ ]:
# Create streaming training and validation datasets

train_dataset = FineWebDataset(
    token_budget=TRAINING_TOKENS,
    context_length=CONTEXT_LENGTH,
    validation=False
)

val_dataset = FineWebDataset(
    token_budget=VALIDATION_TOKENS,
    context_length=CONTEXT_LENGTH,
    validation=True
)


def dataset_size(ds, token_budget, context_length):
    try:
        return len(ds), ""
    except TypeError:
        return token_budget // context_length, " (estimated)"


n_train, note_train = dataset_size(
    train_dataset,
    TRAINING_TOKENS,
    CONTEXT_LENGTH
)

n_val, note_val = dataset_size(
    val_dataset,
    VALIDATION_TOKENS,
    CONTEXT_LENGTH
)

print(f"Training dataset: {n_train:,} samples{note_train}")
print(f"Validation dataset: {n_val:,} samples{note_val}")

Training dataset: 195,312 samples
Validation dataset: 3,906 samples


Create dataloaders.

In [ ]:
# Create DataLoaders

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    drop_last=True
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Training batches: 24414
Validation batches: 488


Batch sanity check.

In [ ]:
# ============================================================
# Batch sanity check
# ============================================================

inputs, labels = next(iter(train_loader))

print("Input shape:", inputs.shape)
print("Labels shape:", labels.shape)

print("\nFirst 20 input tokens:")
print(inputs[0][:20])

print("\nFirst 20 label tokens:")
print(labels[0][:20])

print("\nDecoded input:")
print(tokenizer.decode(inputs[0].tolist()))

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Input shape: torch.Size([8, 256])
Labels shape: torch.Size([8, 256])

First 20 input tokens:
tensor([ 2437,  3486,  2098,   287,   477, 17519,   422, 33718,    12,  2536,
         5973,  7652, 16192,   807,    11,  2321,   198,  2215,   262,   717])

First 20 label tokens:
tensor([ 3486,  2098,   287,   477, 17519,   422, 33718,    12,  2536,  5973,
         7652, 16192,   807,    11,  2321,   198,  2215,   262,   717,  2726])

Decoded input:
How AP reported in all formats from tornado-stricken regionsMarch 8, 2012
When the first serious bout of tornadoes of 2012 blew through middle America in the middle of the night, they touched down in places hours from any AP bureau. Our closest video journalist was Chicago-based Robert Ray, who dropped his plans to travel to Georgia for Super Tuesday, booked several flights to the cities closest to the strikes and headed for the airport. He’d decide once there which flight to take.
He never got on board a plane. Instead, he ended up driving toward

Model sanity check.

In [ ]:
# ============================================================
# 8. MODEL SANITY CHECK
# ============================================================

model.train()

inputs, labels = next(iter(train_loader))

inputs = inputs.to(device)
labels = labels.to(device)

outputs = model(
    input_ids=inputs,
    labels=labels
)

print("Input shape:", inputs.shape)
print("Labels shape:", labels.shape)
print("Logits shape:", outputs.logits.shape)
print("Initial loss:", outputs.loss.item())

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Input shape: torch.Size([8, 256])
Labels shape: torch.Size([8, 256])
Logits shape: torch.Size([8, 256, 50257])
Initial loss: 10.934560775756836


In [ ]:
# ============================================================
# 8. MODEL SANITY CHECK — BACKWARD + UPDATE
# ============================================================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

model.train()

inputs, labels = next(iter(train_loader))

inputs = inputs.to(device)
labels = labels.to(device)

# Save weights before update
before = model.transformer.wte.weight.detach().clone()

# Forward pass
outputs = model(
    input_ids=inputs,
    labels=labels
)

loss = outputs.loss

# Backward
optimizer.zero_grad(set_to_none=True)
loss.backward()

# Update
optimizer.step()

# Compare weights
after = model.transformer.wte.weight.detach()

weight_change = (after - before).abs().sum().item()

print("Loss:", loss.item())
print("Weight change:", weight_change)

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Loss: 10.944214820861816
Weight change: 7672.29296875


# ============================================================
# 9. PRETRAINING
# ============================================================

Reset model

In [ ]:
# ============================================================
# 9. PRETRAINING
# ============================================================

import os
import math
import contextlib
import torch
from transformers import get_cosine_schedule_with_warmup


# ------------------------------------------------------------
# 9.1 Optimizer and Scheduler
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

steps_per_epoch = math.ceil(
    len(train_loader) / GRADIENT_ACCUMULATION_STEPS
)

total_steps = steps_per_epoch * NUM_EPOCHS

warmup_steps = int(0.05 * total_steps)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

print("Optimizer: AdamW")
print("Steps per epoch:", steps_per_epoch)
print("Total optimizer steps:", total_steps)
print("Warmup steps:", warmup_steps)


# ------------------------------------------------------------
# 9.2 Mixed Precision
# ------------------------------------------------------------

AMP_ENABLED = torch.cuda.is_available() and (
    USE_BF16 or USE_FP16
)

if USE_BF16:
    AMP_DTYPE = torch.bfloat16
elif USE_FP16:
    AMP_DTYPE = torch.float16
else:
    AMP_DTYPE = None

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_FP16
)

print("AMP enabled:", AMP_ENABLED)
print("AMP dtype:", AMP_DTYPE)


# ------------------------------------------------------------
# 9.3 Validation
# ------------------------------------------------------------

def evaluate(model, val_loader, device):

    model.eval()

    total_loss = 0.0

    if AMP_ENABLED:
        autocast_context = torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE
        )
    else:
        autocast_context = contextlib.nullcontext()

    with torch.no_grad():

        for inputs, labels in val_loader:

            inputs = inputs.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            with autocast_context:

                outputs = model(
                    input_ids=inputs,
                    labels=labels
                )

            total_loss += outputs.loss.item()

    return total_loss / len(val_loader)


# ------------------------------------------------------------
# 9.4 Checkpoint Saving
# ------------------------------------------------------------

def save_checkpoint(
    model,
    optimizer,
    scheduler,
    scaler,
    epoch,
    global_step,
    train_loss,
    val_loss,
    path
):

    checkpoint = {
        "epoch": epoch,
        "global_step": global_step,

        "model_state_dict":
            model.state_dict(),

        "optimizer_state_dict":
            optimizer.state_dict(),

        "scheduler_state_dict":
            scheduler.state_dict(),

        "scaler_state_dict":
            scaler.state_dict(),

        "train_loss": train_loss,
        "val_loss": val_loss,

        "config": config.to_dict()
    }

    # Save to temporary file first
    temp_path = path + ".tmp"

    torch.save(
        checkpoint,
        temp_path
    )

    # Replace old checkpoint safely
    os.replace(
        temp_path,
        path
    )


# ------------------------------------------------------------
# 9.5 Training Loop
# ------------------------------------------------------------

def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    scheduler,
    scaler,
    device,
    num_epochs,
    checkpoint_dir
):

    os.makedirs(
        checkpoint_dir,
        exist_ok=True
    )

    global_step = 0
    best_val_loss = float("inf")

    model.train()

    for epoch in range(num_epochs):

        total_loss = 0.0

        optimizer.zero_grad(
            set_to_none=True
        )

        print("\n" + "=" * 60)
        print(
            f"Epoch {epoch + 1}/{num_epochs}"
        )
        print("=" * 60)

        total_batches = len(train_loader)

        for batch_idx, (inputs, labels) in enumerate(
            train_loader
        ):

            inputs = inputs.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            # Determine size of this accumulation group
            group_start = (
                batch_idx
                - batch_idx % GRADIENT_ACCUMULATION_STEPS
            )

            group_size = min(
                GRADIENT_ACCUMULATION_STEPS,
                total_batches - group_start
            )

            # Mixed precision
            if AMP_ENABLED:

                autocast_context = torch.autocast(
                    device_type="cuda",
                    dtype=AMP_DTYPE
                )

            else:

                autocast_context = (
                    contextlib.nullcontext()
                )

            with autocast_context:

                outputs = model(
                    input_ids=inputs,
                    labels=labels
                )

                loss = outputs.loss

            total_loss += loss.item()

            # Correct gradient accumulation scaling
            scaled_loss = (
                loss / group_size
            )

            if USE_FP16:

                scaler.scale(
                    scaled_loss
                ).backward()

            else:

                scaled_loss.backward()

            # Update weights after accumulation
            is_update_step = (
                (batch_idx + 1)
                % GRADIENT_ACCUMULATION_STEPS == 0
                or
                (batch_idx + 1) == total_batches
            )

            if is_update_step:

                # Gradient clipping
                if USE_FP16:

                    scaler.unscale_(
                        optimizer
                    )

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    max_norm=1.0
                )

                # Optimizer update
                if USE_FP16:

                    scaler.step(
                        optimizer
                    )

                    scaler.update()

                else:

                    optimizer.step()

                # Learning-rate scheduler
                scheduler.step()

                # Clear gradients
                optimizer.zero_grad(
                    set_to_none=True
                )

                global_step += 1

            # Progress logging
            if (batch_idx + 1) % 500 == 0:

                avg_loss = (
                    total_loss
                    / (batch_idx + 1)
                )

                current_lr = (
                    optimizer.param_groups[0]["lr"]
                )

                print(
                    f"Batch [{batch_idx + 1}/"
                    f"{total_batches}] "
                    f"Loss: {loss.item():.4f} "
                    f"Avg: {avg_loss:.4f} "
                    f"LR: {current_lr:.2e} "
                    f"Step: {global_step}"
                )

            # Periodic checkpoint
            # Only ONE file is maintained: latest.pt
            if (
                global_step > 0
                and global_step % 1000 == 0
                and is_update_step
            ):

                avg_loss = (
                    total_loss
                    / (batch_idx + 1)
                )

                latest_path = os.path.join(
                    checkpoint_dir,
                    "latest.pt"
                )

                save_checkpoint(
                    model=model,
                    optimizer=optimizer,
                    scheduler=scheduler,
                    scaler=scaler,
                    epoch=epoch + 1,
                    global_step=global_step,
                    train_loss=avg_loss,
                    val_loss=None,
                    path=latest_path
                )

                print(
                    "Checkpoint saved → latest.pt"
                )

        # ----------------------------------------------------
        # End of epoch
        # ----------------------------------------------------

        train_loss = (
            total_loss
            / total_batches
        )

        print(
            f"\nTrain Loss: {train_loss:.4f}"
        )

        # Validation
        val_loss = evaluate(
            model,
            val_loader,
            device
        )

        print(
            f"Val Loss:   {val_loss:.4f}"
        )

        # ----------------------------------------------------
        # Save exact epoch checkpoint
        # ----------------------------------------------------

        epoch_path = os.path.join(
            checkpoint_dir,
            f"epoch_{epoch + 1}.pt"
        )

        save_checkpoint(
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            epoch=epoch + 1,
            global_step=global_step,
            train_loss=train_loss,
            val_loss=val_loss,
            path=epoch_path
        )

        print(
            f"Epoch checkpoint saved → "
            f"epoch_{epoch + 1}.pt"
        )

        # ----------------------------------------------------
        # Update latest checkpoint
        # ----------------------------------------------------

        latest_path = os.path.join(
            checkpoint_dir,
            "latest.pt"
        )

        save_checkpoint(
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            epoch=epoch + 1,
            global_step=global_step,
            train_loss=train_loss,
            val_loss=val_loss,
            path=latest_path
        )

        print(
            "Latest checkpoint updated → latest.pt"
        )

        # ----------------------------------------------------
        # Best model
        # ----------------------------------------------------

        if val_loss < best_val_loss:

            best_val_loss = val_loss

            best_path = os.path.join(
                checkpoint_dir,
                "best_model.pt"
            )

            save_checkpoint(
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                scaler=scaler,
                epoch=epoch + 1,
                global_step=global_step,
                train_loss=train_loss,
                val_loss=val_loss,
                path=best_path
            )

            print(
                "⭐ New best model → best_model.pt"
            )

    print("\nTraining completed.")

    return model

Optimizer: AdamW
Steps per epoch: 6104
Total optimizer steps: 6104
Warmup steps: 305
AMP enabled: True
AMP dtype: torch.bfloat16


Starting training.

In [ ]:
# ============================================================
# START PRETRAINING
# ============================================================

model = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    scaler=scaler,
    device=device,
    num_epochs=NUM_EPOCHS,
    checkpoint_dir=CHECKPOINT_DIR
)


Epoch 1/1


Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1848 > 1024). Running this sequence through the model will result in indexing errors


Batch [500/24414] Loss: 7.9390 Avg: 9.0697 LR: 1.23e-04 Step: 125
Batch [1000/24414] Loss: 7.2627 Avg: 8.3470 LR: 2.46e-04 Step: 250
Batch [1500/24414] Loss: 7.0941 Avg: 8.0327 LR: 3.00e-04 Step: 375
Batch [2000/24414] Loss: 7.9369 Avg: 7.8830 LR: 2.99e-04 Step: 500
Batch [2500/24414] Loss: 7.5811 Avg: 7.7865 LR: 2.98e-04 Step: 625
Batch [3000/24414] Loss: 7.3567 Avg: 7.6976 LR: 2.96e-04 Step: 750
Batch [3500/24414] Loss: 6.9422 Avg: 7.6326 LR: 2.93e-04 Step: 875
Batch [4000/24414] Loss: 6.4652 Avg: 7.5749 LR: 2.89e-04 Step: 1000
Checkpoint saved → latest.pt
Batch [4500/24414] Loss: 6.9272 Avg: 7.5162 LR: 2.85e-04 Step: 1125
Batch [5000/24414] Loss: 6.7374 Avg: 7.4648 LR: 2.81e-04 Step: 1250
Batch [5500/24414] Loss: 6.1797 Avg: 7.4196 LR: 2.75e-04 Step: 1375
Batch [6000/24414] Loss: 6.9412 Avg: 7.3855 LR: 2.70e-04 Step: 1500
Batch [6500/24414] Loss: 7.1518 Avg: 7.3590 LR: 2.63e-04 Step: 1625
Batch [7000/24414] Loss: 5.3998 Avg: 7.3275 LR: 2.56e-04 Step: 1750
Batch [7500/24414] Loss: 7.

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Val Loss:   6.4932
Epoch checkpoint saved → epoch_1.pt
Latest checkpoint updated → latest.pt
⭐ New best model → best_model.pt

Training completed.


In [ ]:
# ============================================================
# 10. TEXT GENERATION
# ============================================================

best_checkpoint_path = os.path.join(
    CHECKPOINT_DIR,
    "best_model.pt"
)

checkpoint = torch.load(
    best_checkpoint_path,
    map_location=device
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model.to(device)
model.eval()

print("Loaded best model.")
print("Epoch:", checkpoint["epoch"])
print("Validation loss:", checkpoint["val_loss"])

Loaded best model.
Epoch: 1
Validation loss: 6.493157455178558


In [ ]:
def generate_text(
    prompt,
    max_new_tokens=80,
    temperature=0.8,
    top_k=50
):
    model.eval()

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    input_ids = inputs["input_ids"].to(device)

    with torch.no_grad():

        for _ in range(max_new_tokens):

            # Keep within model context length
            input_ids_cond = input_ids[:, -CONTEXT_LENGTH:]

            outputs = model(
                input_ids=input_ids_cond
            )

            logits = outputs.logits[:, -1, :]

            # Temperature
            logits = logits / temperature

            # Top-k sampling
            if top_k is not None:

                values, indices = torch.topk(
                    logits,
                    min(top_k, logits.size(-1))
                )

                filtered_logits = torch.full_like(
                    logits,
                    float("-inf")
                )

                filtered_logits.scatter_(
                    1,
                    indices,
                    values
                )

                logits = filtered_logits

            probabilities = torch.softmax(
                logits,
                dim=-1
            )

            next_token = torch.multinomial(
                probabilities,
                num_samples=1
            )

            input_ids = torch.cat(
                [input_ids, next_token],
                dim=1
            )

    return tokenizer.decode(
        input_ids[0],
        skip_special_tokens=True
    )

In [ ]:
prompts = [
    "Hello, how are you?",
    "I am going to the store because",
    "The weather today is",
    "My favorite thing to do is",
    "Can you help me with",
    "Yesterday I went to",
    "I think that learning English is"
]

for prompt in prompts:

    print("\n" + "=" * 60)
    print("PROMPT:", prompt)
    print("=" * 60)

    print(
        generate_text(
            prompt,
            max_new_tokens=60,
            temperature=0.8,
            top_k=50
        )
    )


PROMPT: Hello, how are you?
Hello, how are you?What you going be? you't to to all things to out and out you the, the goes you!I love go. I, you just not what�s of. it� the, your like know you� going be.,'m,. do think�
 think are, you

PROMPT: I am going to the store because
I am going to the store because I it� been. the is great I.�m sure don� think can do go it the, a---.�
 had very in. got get, you. of a of to where had in middle a to, you� been. don� like get, can�

PROMPT: The weather today is
The weather today is the, its part a for project to their of.It� easy to the of, it� a and of to of to and, you find your and your.The will be to you And an amount you.The system be to the of you to the of and it the, you use

PROMPT: My favorite thing to do is
My favorite thing to do is I my. did and about. was a. had a minutes at level the, was I, it not that. I out my and got all the, would my and to a more. did put up the. is and it me the was, it a good but was. knew I go

PROMPT: Can 

In [ ]:
# ============================================================
# 11. DAILYDIALOG SUPERVISED FINE-TUNING (SFT)
# ============================================================

# Load the best FineWeb-pretrained model

PRETRAINED_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "best_model.pt"
)

checkpoint = torch.load(
    PRETRAINED_CHECKPOINT,
    map_location=device,
    weights_only=False
)

model.load_state_dict(
    checkpoint["model_state_dict"]
)

model = model.to(device)
model.config.use_cache = False

print("Pretrained model loaded!")
print("Pretraining epoch:", checkpoint["epoch"])
print("Pretraining validation loss:", checkpoint["val_loss"])
print(
    "Parameters:",
    sum(p.numel() for p in model.parameters())
)

Pretrained model loaded!
Pretraining epoch: 1
Pretraining validation loss: 6.493157455178558
Parameters: 51082752


In [ ]:
# ------------------------------------------------------------
# Load conversational dataset
# ------------------------------------------------------------

from datasets import load_dataset

dialogue_dataset = load_dataset(
    "OpenRL/daily_dialog"
)

print(dialogue_dataset)

print("\nExample conversation:")
print(
    "\n".join(
        dialogue_dataset["train"][0]["dialog"]
    )
)

README.md:   0%|          | 0.00/892 [00:00<?, ?B/s]

data/train-00000-of-00001-f151c79abb2c1f(…): reconstructing file:   0%|          |  0.00B / 3.61MB            

data/train-00000-of-00001-f151c79abb2c1f(…): downloading bytes:           |  0.00B            

data/validation-00000-of-00001-2407eb323(…): reconstructing file:   0%|          |  0.00B /  334kB            

data/validation-00000-of-00001-2407eb323(…): downloading bytes:           |  0.00B            

...

data/test-00000-of-00001-66dc7d981b70c91(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/11118 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['dialog', 'act', 'emotion'],
        num_rows: 11118
    })
    validation: Dataset({
        features: ['dialog', 'act', 'emotion'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['dialog', 'act', 'emotion'],
        num_rows: 1000
    })
})

Example conversation:
Say , Jim , how about going for a few beers after dinner ? 
 You know that is tempting but is really not good for our fitness . 
 What do you mean ? It will help us to relax . 
 Do you really think so ? I don't . It will just make us fat and act silly . Remember last time ? 
 I guess you are right.But what shall we do ? I don't feel like sitting at home . 
 I suggest a walk over to the gym where we can play singsong and meet some of our friends . 
 That's a good idea . I hear Mary and Sally often go there to play pingpong.Perhaps we can make a foursome with them . 
 Sounds great to me ! If they are willing , we could ask them to go dancing with us.Th

In [ ]:
import re

def clean_dialogue_text(text: str) -> str:
    # 1. Normalize curly quotes
    text = (text.replace("’", "'").replace("‘", "'")
                .replace("“", '"').replace("”", '"'))

    # 2. Join spaced contractions only: don ' t -> don't, that ' s -> that's
    #    (leaves quoted speech like ' hello ' alone)
    text = re.sub(r"(\w)\s*'\s*(s|t|m|d|ll|re|ve)\b", r"\1'\2", text,
                  flags=re.IGNORECASE)

    # 3. Remove spaces before punctuation: "you ?" -> "you?"
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)

    # 4. Add a space after punctuation when it's glued to the next word
    #    "right.But" -> "right. But", "hi,there" -> "hi, there"
    #    (the lowercase/uppercase checks keep U.S.A intact)
    text = re.sub(r"(?<=[a-z])\.(?=[A-Z])", ". ", text)
    text = re.sub(r"(?<![A-Z.])([,;!?])(?=[A-Za-z])", r"\1 ", text)

    # 5. Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()
    return text


tests = [
    "Hello , how are you ?",
    "I don ' t know .",
    "That ' s right.But what do you think ?",
    "He said ' hello ' to me .",
    "I live in the U.S.A and love it .",
    "Really ? ! I can ’ t believe it .",
]
for t in tests:
    print("Before:", t)
    print("After: ", clean_dialogue_text(t))
    print()

Before: Hello , how are you ?
After:  Hello, how are you?

Before: I don ' t know .
After:  I don't know.

Before: That ' s right.But what do you think ?
After:  That's right. But what do you think?

Before: He said ' hello ' to me .
After:  He said ' hello ' to me.

Before: I live in the U.S.A and love it .
After:  I live in the U.S.A and love it.

Before: Really ? ! I can ’ t believe it .
After:  Really?! I can't believe it.



In [ ]:
# ============================================================
# 11.3 PREPARE SFT EXAMPLES
# ============================================================

from torch.utils.data import Dataset, DataLoader

# GPT-2 uses EOS as the padding token.
# Padding tokens will be ignored during loss calculation.
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id


class DialogueSFTDataset(Dataset):

    def __init__(self, split, tokenizer, max_length=256):

        self.examples = []
        self.tokenizer = tokenizer
        self.max_length = max_length

        for row in split:

            # Extract non-empty conversation turns.
            turns = [
                turn.strip()
                for turn in row["dialog"]
                if turn.strip()
            ]

            # Train on each assistant response.
            # Turns alternate between User and Assistant.
            for target_idx in range(1, len(turns), 2):

                history = turns[:target_idx]
                response = turns[target_idx]

                # ------------------------------------------------
                # Format conversation history
                # ------------------------------------------------

                prompt_parts = []

                for i, utterance in enumerate(history):

                    role = (
                        "User"
                        if i % 2 == 0
                        else "Assistant"
                    )

                    prompt_parts.append(
                        f"{role}: {utterance}\n"
                    )

                prompt = (
                    "".join(prompt_parts)
                    + "Assistant:"
                )

                # ------------------------------------------------
                # Tokenize prompt and target response separately
                # ------------------------------------------------

                prompt_ids = tokenizer(
                    prompt,
                    add_special_tokens=False
                )["input_ids"]

                # Include a leading space for GPT-2's
                # byte-level BPE tokenization.
                response_ids = tokenizer(
                    " " + response,
                    add_special_tokens=False
                )["input_ids"]

                # Mark the end of the target response.
                response_ids.append(
                    tokenizer.eos_token_id
                )

                # ------------------------------------------------
                # Truncate long responses safely
                # ------------------------------------------------

                if len(response_ids) >= max_length:

                    response_ids = response_ids[
                        :max_length - 1
                    ]

                    # Ensure the response ends with EOS.
                    response_ids[-1] = (
                        tokenizer.eos_token_id
                    )

                # ------------------------------------------------
                # Preserve the most recent conversation context
                # ------------------------------------------------

                max_prompt_length = (
                    max_length - len(response_ids)
                )

                prompt_ids = prompt_ids[
                    -max_prompt_length:
                ]

                # Combine prompt and response tokens.
                input_ids = prompt_ids + response_ids

                # ------------------------------------------------
                # Mask prompt tokens during loss calculation
                # ------------------------------------------------

                labels = (
                    [-100] * len(prompt_ids)
                    + response_ids.copy()
                )

                self.examples.append({
                    "input_ids": input_ids,
                    "labels": labels
                })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


print("DialogueSFTDataset defined successfully.")

DialogueSFTDataset defined successfully.


In [ ]:
# ------------------------------------------------------------
# 11.4 Create SFT Datasets
# ------------------------------------------------------------

sft_train_dataset = DialogueSFTDataset(
    dialogue_dataset["train"],
    tokenizer,
    max_length=CONTEXT_LENGTH
)

sft_val_dataset = DialogueSFTDataset(
    dialogue_dataset["validation"],
    tokenizer,
    max_length=CONTEXT_LENGTH
)

print("Training examples:", len(sft_train_dataset))
print("Validation examples:", len(sft_val_dataset))

Training examples: 41637
Validation examples: 3851


In [ ]:
# ------------------------------------------------------------
# 11.5 Batch Collator
# ------------------------------------------------------------

def collate_sft(batch):

    batch_size = len(batch)
    max_len = max(len(item["input_ids"]) for item in batch)

    input_ids = torch.full(
        (batch_size, max_len),
        tokenizer.pad_token_id,
        dtype=torch.long
    )

    labels = torch.full(
        (batch_size, max_len),
        -100,
        dtype=torch.long
    )

    attention_mask = torch.zeros(
        (batch_size, max_len),
        dtype=torch.long
    )

    for i, item in enumerate(batch):

        length = len(item["input_ids"])

        input_ids[i, :length] = torch.tensor(
            item["input_ids"],
            dtype=torch.long
        )

        labels[i, :length] = torch.tensor(
            item["labels"],
            dtype=torch.long
        )

        attention_mask[i, :length] = 1

    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": attention_mask
    }

In [ ]:
# ------------------------------------------------------------
# 11.6 SFT DataLoaders
# ------------------------------------------------------------

SFT_BATCH_SIZE = 8

sft_train_loader = DataLoader(
    sft_train_dataset,
    batch_size=SFT_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_sft
)

sft_val_loader = DataLoader(
    sft_val_dataset,
    batch_size=SFT_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_sft
)

print("Training batches:", len(sft_train_loader))
print("Validation batches:", len(sft_val_loader))

Training batches: 5205
Validation batches: 482


In [ ]:
# ------------------------------------------------------------
# 11.7 SFT Batch Sanity Check
# ------------------------------------------------------------

batch = next(iter(sft_train_loader))

print("Input shape:", batch["input_ids"].shape)
print("Labels shape:", batch["labels"].shape)
print("Attention mask shape:", batch["attention_mask"].shape)

# Display an example prompt + response.
input_ids = batch["input_ids"][0]
labels = batch["labels"][0]

print("\nFull example:")
print(
    tokenizer.decode(
        input_ids.tolist(),
        skip_special_tokens=True
    )
)

# -100 marks tokens excluded from the loss.
target_ids = labels[labels != -100]

print("\nTarget response:")
print(
    tokenizer.decode(
        target_ids.tolist(),
        skip_special_tokens=True
    )
)

Input shape: torch.Size([8, 193])
Labels shape: torch.Size([8, 193])
Attention mask shape: torch.Size([8, 193])

Full example:
User: I'd like to say good-bye to everyone .
Assistant: What time are you going ?

Target response:
 What time are you going ?


In [ ]:
# ============================================================
# 11.8 SFT TRAINING CONFIGURATION
# ============================================================

import math
from transformers import get_cosine_schedule_with_warmup

SFT_CHECKPOINT_DIR = os.path.join(
    PROJECT_DIR,
    "sft_checkpoints"
)

os.makedirs(SFT_CHECKPOINT_DIR, exist_ok=True)

SFT_EPOCHS = 2
SFT_LEARNING_RATE = 5e-5
SFT_GRAD_ACCUM_STEPS = 4

# Create a fresh optimizer for fine-tuning.
sft_optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=SFT_LEARNING_RATE,
    weight_decay=0.01
)

sft_steps_per_epoch = math.ceil(
    len(sft_train_loader) / SFT_GRAD_ACCUM_STEPS
)

sft_total_steps = sft_steps_per_epoch * SFT_EPOCHS

sft_warmup_steps = int(0.05 * sft_total_steps)

sft_scheduler = get_cosine_schedule_with_warmup(
    sft_optimizer,
    num_warmup_steps=sft_warmup_steps,
    num_training_steps=sft_total_steps
)

sft_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_FP16
)

model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id

print("SFT epochs:", SFT_EPOCHS)
print("Learning rate:", SFT_LEARNING_RATE)
print("Training examples:", len(sft_train_dataset))
print("Optimizer steps:", sft_total_steps)
print("Checkpoint directory:", SFT_CHECKPOINT_DIR)

SFT epochs: 2
Learning rate: 5e-05
Training examples: 41637
Optimizer steps: 2604
Checkpoint directory: /content/drive/MyDrive/tiny-GPT-v2/sft_checkpoints


In [ ]:
# ============================================================
# 11.9 SFT EVALUATION + CHECKPOINTS
# ============================================================

def evaluate_sft(model, loader):

    model.eval()
    total_loss = 0.0

    with torch.no_grad():

        for batch in loader:

            input_ids = batch["input_ids"].to(device)
            labels = batch["labels"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            if AMP_ENABLED:
                amp_context = torch.autocast(
                    device_type="cuda",
                    dtype=AMP_DTYPE
                )
            else:
                amp_context = contextlib.nullcontext()

            with amp_context:
                outputs = model(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels
                )

            total_loss += outputs.loss.item()

    model.train()

    return total_loss / len(loader)


def save_sft_checkpoint(
    path,
    epoch,
    train_loss,
    val_loss
):

    checkpoint = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": sft_optimizer.state_dict(),
        "scheduler_state_dict": sft_scheduler.state_dict(),
        "scaler_state_dict": sft_scaler.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "config": config.to_dict()
    }

    temp_path = path + ".tmp"

    torch.save(checkpoint, temp_path)
    os.replace(temp_path, path)

    print("Saved:", path)

In [ ]:
# ============================================================
# FIX: MIXED PRECISION CONFIGURATION FOR SFT
# ============================================================

AMP_ENABLED = torch.cuda.is_available() and (
    USE_BF16 or USE_FP16
)

if USE_BF16:
    AMP_DTYPE = torch.bfloat16
elif USE_FP16:
    AMP_DTYPE = torch.float16
else:
    AMP_DTYPE = None

sft_scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_FP16
)

print("AMP enabled:", AMP_ENABLED)
print("AMP dtype:", AMP_DTYPE)
print("FP16 scaler enabled:", USE_FP16)

AMP enabled: True
AMP dtype: torch.bfloat16
FP16 scaler enabled: False


In [ ]:
# ============================================================
# 11.10 START SUPERVISED FINE-TUNING
# ============================================================

best_sft_val_loss = float("inf")

for epoch in range(SFT_EPOCHS):

    model.train()
    total_loss = 0.0

    sft_optimizer.zero_grad(set_to_none=True)

    print(f"\n{'=' * 55}")
    print(f"SFT Epoch {epoch + 1}/{SFT_EPOCHS}")
    print(f"{'=' * 55}")

    total_batches = len(sft_train_loader)

    for batch_idx, batch in enumerate(sft_train_loader):

        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        # Handle incomplete gradient-accumulation groups.
        group_start = (
            batch_idx // SFT_GRAD_ACCUM_STEPS
        ) * SFT_GRAD_ACCUM_STEPS

        group_size = min(
            SFT_GRAD_ACCUM_STEPS,
            total_batches - group_start
        )

        if AMP_ENABLED:
            amp_context = torch.autocast(
                device_type="cuda",
                dtype=AMP_DTYPE
            )
        else:
            amp_context = contextlib.nullcontext()

        with amp_context:

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss

        total_loss += loss.item()

        scaled_loss = loss / group_size

        if USE_FP16:
            sft_scaler.scale(scaled_loss).backward()
        else:
            scaled_loss.backward()

        update_step = (
            (batch_idx + 1) % SFT_GRAD_ACCUM_STEPS == 0
            or (batch_idx + 1) == total_batches
        )

        if update_step:

            if USE_FP16:
                sft_scaler.unscale_(sft_optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            if USE_FP16:
                sft_scaler.step(sft_optimizer)
                sft_scaler.update()
            else:
                sft_optimizer.step()

            sft_scheduler.step()
            sft_optimizer.zero_grad(set_to_none=True)

        if (batch_idx + 1) % 200 == 0:

            print(
                f"Batch [{batch_idx + 1}/{total_batches}] "
                f"Loss: {loss.item():.4f} "
                f"Average: {total_loss / (batch_idx + 1):.4f}"
            )

    train_loss = total_loss / total_batches

    val_loss = evaluate_sft(
        model,
        sft_val_loader
    )

    print(f"\nTraining loss:   {train_loss:.4f}")
    print(f"Validation loss: {val_loss:.4f}")

    # Save every completed epoch.
    epoch_path = os.path.join(
        SFT_CHECKPOINT_DIR,
        f"sft_epoch_{epoch + 1}.pt"
    )

    save_sft_checkpoint(
        epoch_path,
        epoch + 1,
        train_loss,
        val_loss
    )

    # Maintain a separate best-validation checkpoint.
    if val_loss < best_sft_val_loss:

        best_sft_val_loss = val_loss

        best_path = os.path.join(
            SFT_CHECKPOINT_DIR,
            "best_sft_model.pt"
        )

        save_sft_checkpoint(
            best_path,
            epoch + 1,
            train_loss,
            val_loss
        )

        print("Best SFT checkpoint updated!")

print("\nSFT training completed.")


SFT Epoch 1/2
Batch [200/5205] Loss: 4.2859 Average: 3.7967
Batch [400/5205] Loss: 4.1361 Average: 3.7780
Batch [600/5205] Loss: 3.2247 Average: 3.7715
Batch [800/5205] Loss: 3.8257 Average: 3.7778
Batch [1000/5205] Loss: 3.7174 Average: 3.7823
Batch [1200/5205] Loss: 3.6330 Average: 3.7831
Batch [1400/5205] Loss: 4.0228 Average: 3.7802
Batch [1600/5205] Loss: 3.7343 Average: 3.7803
Batch [1800/5205] Loss: 4.1342 Average: 3.7753
Batch [2000/5205] Loss: 3.9365 Average: 3.7784
Batch [2200/5205] Loss: 3.3579 Average: 3.7787
Batch [2400/5205] Loss: 3.1362 Average: 3.7748
Batch [2600/5205] Loss: 3.7666 Average: 3.7700
Batch [2800/5205] Loss: 4.5007 Average: 3.7695
Batch [3000/5205] Loss: 4.7749 Average: 3.7684
Batch [3200/5205] Loss: 4.1021 Average: 3.7676
Batch [3400/5205] Loss: 3.8603 Average: 3.7641
Batch [3600/5205] Loss: 3.9450 Average: 3.7624
Batch [3800/5205] Loss: 3.5746 Average: 3.7606
Batch [4000/5205] Loss: 3.3811 Average: 3.7571
Batch [4200/5205] Loss: 3.3824 Average: 3.7534
Ba

In [ ]:
# ============================================================
# 12. LOAD BEST FINE-TUNED MODEL
# ============================================================

checkpoint_path = os.path.join(
    SFT_CHECKPOINT_DIR,
    "best_sft_model.pt"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])

model.to(device)
model.eval()
model.config.use_cache = True

print("Best SFT model loaded!")
print("Epoch:", checkpoint["epoch"])
print("Validation loss:", checkpoint["val_loss"])

Best SFT model loaded!
Epoch: 2
Validation loss: 3.5890897104354322


In [ ]:
# ============================================================
# 13. TEST CLEANED TINYGPT
# ============================================================

prompts = [
    "Hello, how are you?",
    "What do you like to do in your free time?",
    "Would you like to go for a walk?",
    "I'm feeling sad today.",
    "Can you help me with something?",
    "What did you do yesterday?",
    "Do you like music?",
]
import re

def clean_reply(text: str) -> str:
    text = text.replace("’", "'").replace("‘", "'")
    text = re.sub(r"(\w)\s*'\s*(s|t|m|d|ll|re|ve)\b", r"\1'\2", text, flags=re.IGNORECASE)
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:1].upper() + text[1:]
import torch, random
import numpy as np

def set_seed(s=42):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

set_seed(42)   # call this immediately before the prompts loop

for prompt in prompts:
    ...
for prompt in prompts:
    print("\n" + "=" * 50)
    print("USER:", prompt)

    response = generate_reply(
        prompt,
        max_new_tokens=30,
        temperature=0.5,
        top_p=0.9,
        repetition_penalty=1.1
    )
    response = clean_reply(response)
    print("TINYGPT:", response)


USER: Hello, how are you?
TINYGPT: I'm not sure.

USER: What do you like to do in your free time?
TINYGPT: I want to.

USER: Would you like to go for a walk?
TINYGPT: I'm going to be there.

USER: I'm feeling sad today.
TINYGPT: You're right.

USER: Can you help me with something?
TINYGPT: I'm looking for a new house.

USER: What did you do yesterday?
TINYGPT: I was a good idea.

USER: Do you like music?
TINYGPT: I like music.


Pretraining stage 2.

In [ ]:
from pathlib import Path

BASE_DIR = Path("/content/drive/MyDrive/tiny-GPT-v2")

STAGE2_DIR = BASE_DIR / "pretraining_stage2"
STAGE2_CHECKPOINT_DIR = STAGE2_DIR / "checkpoints"
STAGE2_LOG_DIR = STAGE2_DIR / "logs"

for folder in [
    STAGE2_CHECKPOINT_DIR,
    STAGE2_LOG_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Stage 2 directories created:\n")

for folder in [STAGE2_DIR, STAGE2_CHECKPOINT_DIR, STAGE2_LOG_DIR]:
    print("✅", folder)

Stage 2 directories created:

✅ /content/drive/MyDrive/tiny-GPT-v2/pretraining_stage2
✅ /content/drive/MyDrive/tiny-GPT-v2/pretraining_stage2/checkpoints
✅ /content/drive/MyDrive/tiny-GPT-v2/pretraining_stage2/logs


In [ ]:
# ============================================================
# STAGE 2 — PREFLIGHT CHECK
# ============================================================

import os
import torch
from pathlib import Path
from google.colab import drive

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is not connected. Stop here and check Runtime settings."
    )

print("GPU:", torch.cuda.get_device_name(0))
print(
    "GPU memory:",
    round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3,
        2
    ),
    "GB"
)

# Mount persistent Google Drive
drive.mount("/content/drive")

# Original project — preserve these checkpoints
BASE_DIR = Path("/content/drive/MyDrive/tiny-GPT-v2")
ORIGINAL_CHECKPOINT = BASE_DIR / "checkpoints" / "latest.pt"
ORIGINAL_BEST = BASE_DIR / "checkpoints" / "best_model.pt"
OLD_SFT_CHECKPOINT = BASE_DIR / "sft_checkpoints" / "best_sft_model.pt"

# Stage 2 gets its own storage
STAGE2_DIR = BASE_DIR / "pretraining_stage2"
STAGE2_CHECKPOINT_DIR = STAGE2_DIR / "checkpoints"
STAGE2_LOG_DIR = STAGE2_DIR / "logs"

STAGE2_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
STAGE2_LOG_DIR.mkdir(parents=True, exist_ok=True)

print("\nOriginal latest checkpoint exists:", ORIGINAL_CHECKPOINT.exists())
print("Original best checkpoint exists:", ORIGINAL_BEST.exists())
print("Previous SFT checkpoint exists:", OLD_SFT_CHECKPOINT.exists())

if not ORIGINAL_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Original checkpoint not found: {ORIGINAL_CHECKPOINT}"
    )

print(
    "\nOriginal checkpoint size:",
    round(ORIGINAL_CHECKPOINT.stat().st_size / 1024**2, 1),
    "MB"
)

# Inspect the checkpoint without moving it onto the GPU
base_checkpoint = torch.load(
    ORIGINAL_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

print("\n========== CHECKPOINT DETAILS ==========")
print("Global step:", base_checkpoint.get("global_step"))
print("Epoch:", base_checkpoint.get("epoch"))
print("Training loss:", base_checkpoint.get("train_loss"))
print("Validation loss:", base_checkpoint.get("val_loss"))
print("Checkpoint keys:", list(base_checkpoint.keys()))

optimizer_state = base_checkpoint.get("optimizer_state_dict", {})
if optimizer_state:
    print(
        "Saved optimizer LR:",
        [group["lr"] for group in optimizer_state["param_groups"]]
    )

scheduler_state = base_checkpoint.get("scheduler_state_dict", {})
if scheduler_state:
    print(
        "Saved scheduler last LR:",
        scheduler_state.get("_last_lr")
    )

print("\nStage 2 directory:", STAGE2_DIR)
print("Preflight completed successfully.")

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Original latest checkpoint exists: True
Original best checkpoint exists: True
Previous SFT checkpoint exists: True

Original checkpoint size: 584.7 MB

========== CHECKPOINT DETAILS ==========
Global step: 6104
Epoch: 1
Training loss: 6.848579432812567
Validation loss: 6.493157455178558
Checkpoint keys: ['epoch', 'global_step', 'model_state_dict', 'optimizer_state_dict', 'scheduler_state_dict', 'scaler_state_dict', 'train_loss', 'val_loss', 'config']
Saved optimizer LR: [0.0]
Saved scheduler last LR: [0.0]

Stage 2 directory: /content/drive/MyDrive/tiny-GPT-v2/pretraining_stage2
Preflight completed successfully.


In [ ]:
# ============================================================
# TinyGPT v2 — Corrected Stage 2 Pretraining
# 50M tokens | FineWeb-Edu | FP16 | Tesla T4
# ============================================================

import os
import math
import time
import hashlib
from pathlib import Path

import torch
from torch.utils.data import IterableDataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, GPT2Config, GPT2LMHeadModel

assert torch.cuda.is_available(), "Connect to a GPU runtime first."
device = torch.device("cuda")

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

BASE_DIR = Path("/content/drive/MyDrive/tiny-GPT-v2")
BASE_CHECKPOINT = BASE_DIR / "checkpoints" / "latest.pt"

# A separate folder protects the original checkpoints and
# avoids resuming the earlier, incorrectly aligned Stage 2 run.
RUN_DIR = BASE_DIR / "pretraining_stage2_final"
RUN_DIR.mkdir(parents=True, exist_ok=True)

LATEST_PATH = RUN_DIR / "pretrain2_latest.pt"
BEST_PATH = RUN_DIR / "pretrain2_best.pt"

TARGET_TOKENS = 50_000_000
BATCH_SIZE = 8
GRAD_ACCUM = 4
PEAK_LR = 3e-4
MIN_LR = 3e-5
WARMUP_STEPS = 100
WEIGHT_DECAY = 0.01
SAVE_EVERY = 250
LOG_EVERY = 50
EVAL_EVERY = 1000
VAL_TOKENS = 250_000

DATASET_NAME = "HuggingFaceFW/fineweb-edu"
DATASET_CONFIG = "sample-10BT"
SHUFFLE_SEED = 20261013

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

torch.backends.cuda.matmul.allow_tf32 = True

print("GPU:", torch.cuda.get_device_name(0))
print("Stage 2 output:", RUN_DIR)
print("Precision: FP16 with GradScaler")


# ------------------------------------------------------------
# 2. DATASET
# ------------------------------------------------------------

def is_validation_text(text):
    value = int(hashlib.md5(text.encode("utf-8")).hexdigest(), 16)
    return value % 100 < 5


class TextChunks(IterableDataset):
    def __init__(
        self, name, config_name, token_budget, context_length,
        validation=False, shuffle_seed=None
    ):
        self.name = name
        self.config_name = config_name
        self.num_examples = token_budget // context_length
        self.context_length = context_length
        self.validation = validation
        self.shuffle_seed = shuffle_seed

    def __len__(self):
        return self.num_examples

    def __iter__(self):
        kwargs = {"split": "train", "streaming": True}
        if self.config_name:
            kwargs["name"] = self.config_name

        ds = load_dataset(self.name, **kwargs)

        if self.shuffle_seed is not None:
            ds = ds.shuffle(
                seed=self.shuffle_seed,
                buffer_size=10_000
            )

        buffer = []
        produced = 0

        for row in ds:
            text = str(row.get("text", "")).strip()
            if not text:
                continue

            if is_validation_text(text) != self.validation:
                continue

            ids = tokenizer(text, add_special_tokens=False)["input_ids"]
            if not ids:
                continue

            buffer.extend(ids)
            buffer.append(tokenizer.eos_token_id)

            while len(buffer) >= self.context_length:
                chunk = buffer[:self.context_length]
                buffer = buffer[self.context_length:]

                # Important: return only input tokens.
                # GPT2LMHeadModel shifts labels internally.
                yield torch.tensor(chunk, dtype=torch.long)

                produced += 1
                if produced >= self.num_examples:
                    return


# ------------------------------------------------------------
# 3. LOAD MODEL WEIGHTS
# ------------------------------------------------------------

if LATEST_PATH.exists():
    print("Resuming corrected Stage 2 checkpoint.")
    ckpt = torch.load(LATEST_PATH, map_location="cpu", weights_only=False)
    resuming = True
else:
    print("Loading original latest.pt.")
    ckpt = torch.load(BASE_CHECKPOINT, map_location="cpu", weights_only=False)
    resuming = False

base_global_step = int(
    ckpt.get("base_global_step", ckpt.get("global_step", 6104))
)
stage2_step = int(ckpt.get("stage2_step", 0))
micro_batches_seen = int(ckpt.get("micro_batches_seen", 0))
tokens_seen = int(ckpt.get("tokens_seen", 0))

config = GPT2Config.from_dict(ckpt["config"])
config.pad_token_id = tokenizer.pad_token_id
config.use_cache = False

assert config.vocab_size == tokenizer.vocab_size

model = GPT2LMHeadModel(config)
model.load_state_dict(ckpt["model_state_dict"], strict=True)
model.to(device)

print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Original global step:", base_global_step)
print("Stage 2 step:", stage2_step)


# ------------------------------------------------------------
# 4. OPTIMIZER AND FP16
# ------------------------------------------------------------

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=PEAK_LR,
    weight_decay=WEIGHT_DECAY
)

scaler = torch.amp.GradScaler("cuda", enabled=True)

if resuming:
    # Restore states saved by THIS corrected Stage 2 loop.
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    if ckpt.get("scaler_state_dict"):
        scaler.load_state_dict(ckpt["scaler_state_dict"])
    print("Corrected Stage 2 optimizer restored.")
else:
    # Intentionally reset old AdamW moments: they came from
    # the earlier, incorrectly shifted training objective.
    print("Using fresh AdamW state with pretrained model weights.")

corrected_base_val = ckpt.get("corrected_base_val")
best_val_loss = float(ckpt.get("best_val_loss", float("inf")))
last_val_loss = float(ckpt.get("val_loss", float("nan")))
last_eval_step = int(ckpt.get("last_eval_step", 0))
del ckpt


# ------------------------------------------------------------
# 5. STREAMS AND LOADERS
# ------------------------------------------------------------

context_length = config.n_positions

train_ds = TextChunks(
    DATASET_NAME, DATASET_CONFIG, TARGET_TOKENS,
    context_length, validation=False, shuffle_seed=SHUFFLE_SEED
)

# Validation remains on the original FineWeb held-out split.
val_ds = TextChunks(
    "HuggingFaceFW/fineweb", None, VAL_TOKENS,
    context_length, validation=True
)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, num_workers=0,
    pin_memory=True, drop_last=True
)

val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, num_workers=0,
    pin_memory=True, drop_last=True
)

total_batches = len(train_loader)
total_steps = math.ceil(total_batches / GRAD_ACCUM)
tokens_per_step = BATCH_SIZE * context_length * GRAD_ACCUM

print("Training micro-batches:", f"{total_batches:,}")
print("Optimizer steps:", f"{total_steps:,}")
print("Tokens per full optimizer step:", f"{tokens_per_step:,}")
print("Planned input-token positions:",
      f"{total_batches * BATCH_SIZE * context_length:,}")


# ------------------------------------------------------------
# 6. LEARNING-RATE SCHEDULE
# ------------------------------------------------------------

def get_lr(step):
    if step < WARMUP_STEPS:
        p = (step + 1) / WARMUP_STEPS
        return MIN_LR + (PEAK_LR - MIN_LR) * p

    decay_steps = max(1, total_steps - WARMUP_STEPS)
    p = min(1.0, (step - WARMUP_STEPS + 1) / decay_steps)
    cosine = 0.5 * (1.0 + math.cos(math.pi * p))
    return MIN_LR + (PEAK_LR - MIN_LR) * cosine


# ------------------------------------------------------------
# 7. EVALUATION AND CHECKPOINTS
# ------------------------------------------------------------

@torch.no_grad()
def evaluate():
    model.eval()
    loss_sum = 0.0
    count = 0

    for inputs in val_loader:
        inputs = inputs.to(device, non_blocking=True)
        mask = torch.ones_like(inputs)

        with torch.autocast("cuda", dtype=torch.float16):
            out = model(
                input_ids=inputs,
                attention_mask=mask,
                labels=inputs
            )

        loss_sum += out.loss.float().item()
        count += 1

    model.train()
    return loss_sum / max(count, 1)


def save_checkpoint(path):
    state = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "config": model.config.to_dict(),
        "base_global_step": base_global_step,
        "global_step": base_global_step + stage2_step,
        "stage2_step": stage2_step,
        "micro_batches_seen": micro_batches_seen,
        "tokens_seen": tokens_seen,
        "corrected_base_val": corrected_base_val,
        "best_val_loss": best_val_loss,
        "val_loss": last_val_loss,
        "last_eval_step": last_eval_step,
        "lr_config": {
            "peak_lr": PEAK_LR,
            "min_lr": MIN_LR,
            "warmup_steps": WARMUP_STEPS,
            "total_steps": total_steps
        }
    }

    tmp = str(path) + ".tmp"
    torch.save(state, tmp)
    os.replace(tmp, path)
    print(f"Saved {path.name} at Stage 2 step {stage2_step:,}")


# Measure the starting model using the corrected next-token objective.
model.train()
if corrected_base_val is None:
    print("\nMeasuring corrected baseline validation loss...")
    corrected_base_val = evaluate()
    print(f"Corrected baseline validation loss: {corrected_base_val:.4f}")


# ------------------------------------------------------------
# 8. TRAINING LOOP
# ------------------------------------------------------------

model.train()
optimizer.zero_grad(set_to_none=True)

iterator = iter(train_loader)

# Resume from the last saved data position if needed.
for _ in range(micro_batches_seen):
    next(iterator)

window_loss = torch.zeros((), device=device)
window_batches = 0
window_tokens = 0
timer = time.perf_counter()
last_train_loss = float("nan")

print("\nStarting corrected Stage 2 training...")
print(f"LR warmup: {WARMUP_STEPS} steps; cosine floor: {MIN_LR:g}")

for batch_idx in range(micro_batches_seen, total_batches):
    inputs = next(iterator).to(device, non_blocking=True)
    mask = torch.ones_like(inputs)

    micro_batches_seen = batch_idx + 1
    tokens_seen += inputs.numel()
    window_tokens += inputs.numel()

    group_start = (batch_idx // GRAD_ACCUM) * GRAD_ACCUM
    group_size = min(GRAD_ACCUM, total_batches - group_start)

    with torch.autocast("cuda", dtype=torch.float16):
        out = model(
            input_ids=inputs,
            attention_mask=mask,
            labels=inputs
        )
        loss = out.loss

    # Only read the accumulated GPU loss when logging.
    window_loss += loss.detach().float()
    window_batches += 1

    scaler.scale(loss / group_size).backward()

    update_step = (
        (batch_idx + 1) % GRAD_ACCUM == 0
        or (batch_idx + 1) == total_batches
    )

    if not update_step:
        continue

    lr = get_lr(stage2_step)
    for group in optimizer.param_groups:
        group["lr"] = lr

    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

    stage2_step += 1

    if stage2_step % LOG_EVERY == 0:
        elapsed = max(time.perf_counter() - timer, 1e-6)
        last_train_loss = window_loss.item() / max(window_batches, 1)
        speed = window_tokens / elapsed

        print(
            f"Stage2 {stage2_step:,}/{total_steps:,} | "
            f"Global {base_global_step + stage2_step:,} | "
            f"Tokens {tokens_seen:,}/{TARGET_TOKENS:,} | "
            f"Loss {last_train_loss:.4f} | "
            f"LR {lr:.2e} | Speed {speed:,.0f} tok/s"
        )

        window_loss.zero_()
        window_batches = 0
        window_tokens = 0
        timer = time.perf_counter()

    # Validate every 1,000 optimizer steps.
    if stage2_step % EVAL_EVERY == 0:
        last_val_loss = evaluate()
        last_eval_step = stage2_step
        print(
            f"Validation loss: {last_val_loss:.4f} "
            f"(corrected baseline: {corrected_base_val:.4f})"
        )

        if last_val_loss < best_val_loss:
            best_val_loss = last_val_loss
            save_checkpoint(BEST_PATH)

        save_checkpoint(LATEST_PATH)

    elif stage2_step % SAVE_EVERY == 0:
        save_checkpoint(LATEST_PATH)


# ------------------------------------------------------------
# 9. FINAL SAVE AND VALIDATION
# ------------------------------------------------------------

if last_eval_step != stage2_step:
    last_val_loss = evaluate()
    last_eval_step = stage2_step
    print(f"Final validation loss: {last_val_loss:.4f}")

    if last_val_loss < best_val_loss:
        best_val_loss = last_val_loss
        save_checkpoint(BEST_PATH)

save_checkpoint(LATEST_PATH)

print("\nStage 2 training finished.")
print("Input-token positions processed:", f"{tokens_seen:,}")
print("Stage 2 optimizer steps:", f"{stage2_step:,}")
print("Corrected baseline validation:", corrected_base_val)
print("Final validation:", last_val_loss)
print("Latest checkpoint:", LATEST_PATH)
print("Best Stage 2 checkpoint:", BEST_PATH)

GPU: Tesla T4
Stage 2 output: /content/drive/MyDrive/tiny-GPT-v2/pretraining_stage2_final
Precision: FP16 with GradScaler
Loading original latest.pt.
Parameters: 51,082,752
Original global step: 6104
Stage 2 step: 0
Using fresh AdamW state with pretrained model weights.
Training micro-batches: 24,414
Optimizer steps: 6,104
Tokens per full optimizer step: 8,192
Planned input-token positions: 49,999,872

Measuring corrected baseline validation loss...


Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1346 > 1024). Running this sequence through the model will result in indexing errors


Corrected baseline validation loss: 7.1034

Starting corrected Stage 2 training...
LR warmup: 100 steps; cosine floor: 3e-05


...

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Validation loss: 5.0124 (corrected baseline: 7.1034)
Saved pretrain2_best.pt at Stage 2 step 6,000
Saved pretrain2_latest.pt at Stage 2 step 6,000
Stage2 6,050/6,104 | Global 12,154 | Tokens 49,561,600/50,000,000 | Loss 4.7435 | LR 3.01e-05 | Speed 2,202 tok/s
Stage2 6,100/6,104 | Global 12,204 | Tokens 49,971,200/50,000,000 | Loss 4.7253 | LR 3.00e-05 | Speed 18,330 tok/s


Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/27468 [00:00<?, ?it/s]

Final validation loss: 5.0101
Saved pretrain2_best.pt at Stage 2 step 6,104
Saved pretrain2_latest.pt at Stage 2 step 6,104

Stage 2 training finished.
Input-token positions processed: 49,999,872
Stage 2 optimizer steps: 6,104
Corrected baseline validation: 7.103354790171639
Final validation: 5.010143491088367
Latest checkpoint: /content/drive/MyDrive/tiny-GPT-v2/pretraining_stage2_final/pretrain2_latest.pt
Best Stage 2 checkpoint: /content/drive/MyDrive/tiny-GPT-v2/pretraining_stage2_final/pretrain2_best.pt


In [ ]:
from pathlib import Path
import torch

BASE_DIR = Path("/content/drive/MyDrive/tiny-GPT-v2")
STAGE2_DIR = BASE_DIR / "pretraining_stage2_final"

LATEST_PATH = STAGE2_DIR / "pretrain2_latest.pt"
BEST_PATH = STAGE2_DIR / "pretrain2_best.pt"

assert LATEST_PATH.exists(), "Stage 2 latest checkpoint is missing!"
assert BEST_PATH.exists(), "Stage 2 best checkpoint is missing!"

latest = torch.load(
    LATEST_PATH, map_location="cpu", weights_only=False
)
best = torch.load(
    BEST_PATH, map_location="cpu", weights_only=False
)

print("Latest Stage 2 step:", latest["stage2_step"])
print("Tokens processed:", f'{latest["tokens_seen"]:,}')
print("Best Stage 2 step:", best["stage2_step"])
print("Best validation loss:", best["val_loss"])

# Load the best validation checkpoint into our existing model.
model.load_state_dict(best["model_state_dict"])
model.to(device)
model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id

print("\n✅ Best Stage 2 model loaded for SFT.")

KeyboardInterrupt: 

In [ ]:
# ============================================================
# TINYGPT V2 — STAGE 2 SFT (DailyDialog) — corrected
# Base: Stage 2 pretrained checkpoint | Hardware: Tesla T4, FP16
# ============================================================

import os, re, math, time, random
import numpy as np
import torch

from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import GPT2Config, GPT2LMHeadModel, AutoTokenizer


# ============================================================
# 0. SETTINGS  (edit TIME_BUDGET_MIN to fit your remaining runtime)
# ============================================================

TIME_BUDGET_MIN = 25      # remaining Colab minutes minus ~15; max 25
SFT_EPOCHS = 2
SFT_LR = 1e-4
SFT_MIN_LR = 1e-5
SFT_GRAD_ACCUM = 4
SFT_WEIGHT_DECAY = 0.01
SFT_BATCH_SIZE = 8
WARMUP_STEPS = 50


# ============================================================
# 1. PATHS AND SAFETY CHECKS
# ============================================================

BASE_DIR = Path("/content/drive/MyDrive/tiny-GPT-v2")

CANDIDATES = [
    BASE_DIR / "pretraining_stage2_final" / "pretrain2_best.pt",
    BASE_DIR / "pretraining_stage2_final" / "pretrain2_latest.pt",
]
STAGE2_PATH = next((p for p in CANDIDATES if p.exists()), None)
assert STAGE2_PATH is not None, f"No Stage 2 checkpoint found in: {CANDIDATES}"

SFT_DIR = BASE_DIR / "sft_stage2_checkpoints"      # new folder; old SFT untouched
SFT_DIR.mkdir(parents=True, exist_ok=True)
LATEST_SFT_PATH = SFT_DIR / "sft_latest.pt"
BEST_SFT_PATH = SFT_DIR / "best_sft_model.pt"

assert torch.cuda.is_available(), "Connect to a GPU runtime."
device = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))
print("Using base checkpoint:", STAGE2_PATH.name)


# ============================================================
# 2. TOKENIZER, MODEL, AND STAGE 2 WEIGHTS
# ============================================================

if "tokenizer" not in globals():
    tokenizer = AutoTokenizer.from_pretrained("gpt2")

if "model" not in globals():
    model = GPT2LMHeadModel(GPT2Config(
        vocab_size=50257, n_positions=256, n_embd=512,
        n_layer=8, n_head=8,
        bos_token_id=50256, eos_token_id=50256,
    ))

stage2_ckpt = torch.load(STAGE2_PATH, map_location="cpu", weights_only=False)
model.load_state_dict(stage2_ckpt["model_state_dict"], strict=True)
model.to(device)
model.config.use_cache = False
model.config.pad_token_id = tokenizer.eos_token_id

print("Loaded Stage 2 checkpoint.")
print("Stage 2 step:", stage2_ckpt.get("stage2_step"))
print("Pretraining validation loss:", stage2_ckpt.get("val_loss"))
print("Parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
del stage2_ckpt


# ============================================================
# 3. LOAD DAILYDIALOG
# ============================================================

if "dialogue_dataset" not in globals():
    dialogue_dataset = load_dataset("OpenRL/daily_dialog")
print("DailyDialog loaded.")


# ============================================================
# 4. CLEANING FUNCTIONS
# ============================================================

def clean_dialogue_text(text):
    text = (text.replace("’", "'").replace("‘", "'")
                .replace("“", '"').replace("”", '"'))
    text = re.sub(r"(\w)\s*'\s*(s|t|m|d|ll|re|ve)\b", r"\1'\2", text,
                  flags=re.IGNORECASE)
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"(?<=[a-z])\.(?=[A-Z])", ". ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def clean_reply(text):
    text = text.split("\nUser:")[0]
    text = clean_dialogue_text(text)
    return text[:1].upper() + text[1:]


# ============================================================
# 5. RESPONSE-ONLY TRAINING EXAMPLES
#    Format:  User: ...\nAssistant: ...\n ... Assistant: <reply><EOS>
# ============================================================

tokenizer.pad_token = tokenizer.eos_token
MAX_LENGTH = min(256, model.config.n_positions)


class DialogueSFTDataset(Dataset):

    def __init__(self, split):
        self.examples = []
        for row in split:
            turns = [clean_dialogue_text(t) for t in row["dialog"] if t.strip()]

            for target_idx in range(1, len(turns), 2):
                history = turns[:target_idx]
                response = turns[target_idx]

                prompt = "".join(
                    f"{'User' if i % 2 == 0 else 'Assistant'}: {u}\n"
                    for i, u in enumerate(history)
                ) + "Assistant:"

                prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
                response_ids = tokenizer(" " + response,
                                         add_special_tokens=False)["input_ids"]
                response_ids.append(tokenizer.eos_token_id)

                if len(response_ids) >= MAX_LENGTH:
                    response_ids = response_ids[:MAX_LENGTH - 1]
                    response_ids[-1] = tokenizer.eos_token_id

                max_prompt_len = MAX_LENGTH - len(response_ids)
                prompt_ids = prompt_ids[-max_prompt_len:]

                input_ids = prompt_ids + response_ids
                labels = [-100] * len(prompt_ids) + response_ids.copy()

                assert len(input_ids) == len(labels)
                assert any(x != -100 for x in labels)
                self.examples.append({"input_ids": input_ids, "labels": labels})

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return self.examples[i]


def collate_sft(batch):
    bs = len(batch)
    max_len = max(len(b["input_ids"]) for b in batch)
    input_ids = torch.full((bs, max_len), tokenizer.pad_token_id, dtype=torch.long)
    labels = torch.full((bs, max_len), -100, dtype=torch.long)
    attention_mask = torch.zeros((bs, max_len), dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["input_ids"])
        input_ids[i, :n] = torch.tensor(b["input_ids"])
        labels[i, :n] = torch.tensor(b["labels"])
        attention_mask[i, :n] = 1
    return {"input_ids": input_ids, "labels": labels, "attention_mask": attention_mask}


print("\nPreparing DailyDialog examples...")
sft_train_dataset = DialogueSFTDataset(dialogue_dataset["train"])
sft_val_dataset = DialogueSFTDataset(dialogue_dataset["validation"])

sft_train_loader = DataLoader(sft_train_dataset, batch_size=SFT_BATCH_SIZE,
                              shuffle=True, collate_fn=collate_sft,
                              num_workers=0, pin_memory=True)
sft_val_loader = DataLoader(sft_val_dataset, batch_size=SFT_BATCH_SIZE,
                            shuffle=False, collate_fn=collate_sft,
                            num_workers=0, pin_memory=True)

print("Training examples:", len(sft_train_dataset))
print("Validation examples:", len(sft_val_dataset))
print("Training batches per epoch:", len(sft_train_loader))

# Verify masking
ex = sft_train_dataset[0]
targets = [t for t in ex["labels"] if t != -100]
assert targets[-1] == tokenizer.eos_token_id
print("Masking verified. Example target:",
      tokenizer.decode(targets, skip_special_tokens=True))


# ============================================================
# 6. OPTIMIZER, TIME-AWARE LR, SCALER
# ============================================================

steps_per_epoch = math.ceil(len(sft_train_loader) / SFT_GRAD_ACCUM)
total_sft_steps = steps_per_epoch * SFT_EPOCHS
budget_sec = TIME_BUDGET_MIN * 60

sft_optimizer = torch.optim.AdamW(model.parameters(), lr=SFT_LR,
                                  weight_decay=SFT_WEIGHT_DECAY)
sft_scaler = torch.amp.GradScaler("cuda", enabled=True)   # T4: FP16, not BF16


def lr_at(step, train_start):
    if step < WARMUP_STEPS:
        return SFT_LR * (step + 1) / WARMUP_STEPS
    step_p = (step - WARMUP_STEPS) / max(1, total_sft_steps - WARMUP_STEPS)
    time_p = (time.time() - train_start) / budget_sec
    p = min(1.0, max(step_p, time_p))     # decay ends at whichever limit comes first
    return SFT_MIN_LR + 0.5 * (SFT_LR - SFT_MIN_LR) * (1 + math.cos(math.pi * p))


print("\nSFT configuration")
print("Epochs:", SFT_EPOCHS, "| LR:", SFT_LR, "->", SFT_MIN_LR)
print("Planned optimizer steps:", total_sft_steps)
print("Time budget (min):", TIME_BUDGET_MIN)
print("Checkpoint folder:", SFT_DIR)


# ============================================================
# 7. VALIDATION AND CHECKPOINTING
# ============================================================

@torch.no_grad()
def evaluate_sft():
    model.eval()
    loss_sum, batches = 0.0, 0
    for batch in sft_val_loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        with torch.autocast("cuda", dtype=torch.float16):
            out = model(input_ids=input_ids, attention_mask=attention_mask,
                        labels=labels)
        loss_sum += out.loss.float().item()
        batches += 1
    model.train()
    return loss_sum / max(batches, 1)


def save_sft_checkpoint(path, epoch, global_step, train_loss, val_loss):
    ckpt = {
        "epoch": epoch,
        "global_step": global_step,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": sft_optimizer.state_dict(),
        "scaler_state_dict": sft_scaler.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "config": model.config.to_dict(),
    }
    tmp = str(path) + ".tmp"
    torch.save(ckpt, tmp)
    os.replace(tmp, path)
    print("Saved:", Path(path).name)


# # ============================================================
# # 8. TRAIN SFT
# # ============================================================

# best_sft_val_loss = float("inf")
# sft_global_step = 0
# stop_training = False
# train_start = time.time()

# model.train()

# for epoch in range(SFT_EPOCHS):

#     sft_optimizer.zero_grad(set_to_none=True)

#     epoch_loss_sum = torch.zeros((), device=device)
#     log_loss_sum = torch.zeros((), device=device)
#     epoch_loss_count, log_count = 0, 0
#     total_batches = len(sft_train_loader)
#     epoch_start = time.perf_counter()

#     print(f"\n{'=' * 55}\nSFT Epoch {epoch + 1}/{SFT_EPOCHS}\n{'=' * 55}")

#     for batch_idx, batch in enumerate(sft_train_loader):

#         input_ids = batch["input_ids"].to(device, non_blocking=True)
#         labels = batch["labels"].to(device, non_blocking=True)
#         attention_mask = batch["attention_mask"].to(device, non_blocking=True)

#         group_start = (batch_idx // SFT_GRAD_ACCUM) * SFT_GRAD_ACCUM
#         group_size = min(SFT_GRAD_ACCUM, total_batches - group_start)

#         with torch.autocast("cuda", dtype=torch.float16):
#             out = model(input_ids=input_ids, attention_mask=attention_mask,
#                         labels=labels)
#             loss = out.loss

#         epoch_loss_sum += loss.detach().float()
#         log_loss_sum += loss.detach().float()
#         epoch_loss_count += 1
#         log_count += 1

#         sft_scaler.scale(loss / group_size).backward()

#         update_step = ((batch_idx + 1) % SFT_GRAD_ACCUM == 0
#                        or (batch_idx + 1) == total_batches)

#         if update_step:
#             lr = lr_at(sft_global_step, train_start)
#             for g in sft_optimizer.param_groups:
#                 g["lr"] = lr

#             sft_scaler.unscale_(sft_optimizer)
#             torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
#             sft_scaler.step(sft_optimizer)
#             sft_scaler.update()
#             sft_optimizer.zero_grad(set_to_none=True)
#             sft_global_step += 1

#         if (batch_idx + 1) % 200 == 0:
#             avg = log_loss_sum.item() / max(log_count, 1)
#             print(f"Batch {batch_idx + 1}/{total_batches} | Loss {avg:.4f} | "
#                   f"LR {sft_optimizer.param_groups[0]['lr']:.2e} | "
#                   f"{(time.time() - train_start) / 60:.1f} min")
#             log_loss_sum.zero_()
#             log_count = 0

#         if update_step and sft_global_step % 500 == 0:
#             save_sft_checkpoint(LATEST_SFT_PATH, epoch + 1, sft_global_step,
#                                 epoch_loss_sum.item() / max(epoch_loss_count, 1),
#                                 None)

#         # Stop cleanly (only right after an optimizer update) when time is up
#         if update_step and (time.time() - train_start) >= budget_sec:
#             print("\nTime budget reached. Stopping SFT and saving.")
#             stop_training = True
#             break

#     train_loss = epoch_loss_sum.item() / max(epoch_loss_count, 1)
#     print(f"\nTraining loss: {train_loss:.4f}")
#     print("Running validation...")
#     val_loss = evaluate_sft()
#     print(f"Validation loss: {val_loss:.4f}")
#     print(f"Epoch time: {(time.perf_counter() - epoch_start) / 60:.1f} minutes")

#     save_sft_checkpoint(LATEST_SFT_PATH, epoch + 1, sft_global_step,
#                         train_loss, val_loss)

#     if val_loss < best_sft_val_loss:
#         best_sft_val_loss = val_loss
#         save_sft_checkpoint(BEST_SFT_PATH, epoch + 1, sft_global_step,
#                             train_loss, val_loss)
#         print("✅ Best SFT checkpoint updated.")

#     if stop_training:
#         break

# print("\n" + "=" * 55)
# print("TINYGPT SFT FINISHED")
# print("=" * 55)
# print("Best validation loss:", best_sft_val_loss)
# print("Best checkpoint:", BEST_SFT_PATH)


# ============================================================
# 9. TEST THE NEW MODEL (same 7 prompts, same decoding, fixed seed)
# ============================================================

model.config.use_cache = True
model.eval()


def set_seed(s=42):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


@torch.no_grad()
def generate_reply(user_text, max_new_tokens=30, temperature=0.5,
                   top_p=0.9, repetition_penalty=1.1):
    prompt = f"User: {clean_dialogue_text(user_text)}\nAssistant:"
    ids = tokenizer(prompt, return_tensors="pt",
                    add_special_tokens=False)["input_ids"].to(device)
    out = model.generate(
        input_ids=ids,
        attention_mask=torch.ones_like(ids),
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        repetition_penalty=repetition_penalty,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
    )
    reply = tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
    return clean_reply(reply)


prompts = [
    "Hello, how are you?",
    "What do you like to do in your free time?",
    "Would you like to go for a walk?",
    "I'm feeling sad today.",
    "Can you help me with something?",
    "What did you do yesterday?",
    "Do you like music?",
]

set_seed(42)
lines = []
for p in prompts:
    r = generate_reply(p)
    print("\n" + "=" * 50)
    print("USER:", p)
    print("TINYGPT:", r)
    lines.append(f"USER: {p}\nTINYGPT: {r}\n")

(SFT_DIR / "test_outputs.txt").write_text("\n".join(lines))
print("\nOutputs saved to:", SFT_DIR / "test_outputs.txt")

GPU: Tesla T4
Using base checkpoint: pretrain2_best.pt
Loaded Stage 2 checkpoint.
Stage 2 step: 6104
Pretraining validation loss: 5.010143491088367
Parameters: 51,082,752


README.md:   0%|          | 0.00/892 [00:00<?, ?B/s]

data/train-00000-of-00001-f151c79abb2c1f(…): reconstructing file:   0%|          |  0.00B / 3.61MB            

data/train-00000-of-00001-f151c79abb2c1f(…): downloading bytes:           |  0.00B            

data/validation-00000-of-00001-2407eb323(…): reconstructing file:   0%|          |  0.00B /  334kB            

...

data/test-00000-of-00001-66dc7d981b70c91(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/11118 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

DailyDialog loaded.

Preparing DailyDialog examples...
Training examples: 41637
Validation examples: 3851
Training batches per epoch: 5205
Masking verified. Example target:  You know that is tempting but is really not good for our fitness.

SFT configuration
Epochs: 2 | LR: 0.0001 -> 1e-05
Planned optimizer steps: 2604
Time budget (min): 25
Checkpoint folder: /content/drive/MyDrive/tiny-GPT-v2/sft_stage2_checkpoints

USER: Hello, how are you?
TINYGPT: What is the 'digital'? The main goal of the technology is to build a new technology that will be used to create a �

USER: What do you like to do in your free time?
TINYGPT: The Future of the World's World. New York, DC: The World's National Institute for International Development. The World Bank

USER: Would you like to go for a walk?
TINYGPT: What is the difference between a computer? A computer or a computer, can be used in many different ways. The user's ability to work

USER: I'm feeling sad today.
TINYGPT: A.D., and A.D., J.A., & A.

In [ ]:
seeds = [7, 123, 2024]
all_lines = []

for s in seeds:
    set_seed(s)                      # reseed before each full pass
    header = f"\n######## SEED {s} ########"
    print(header)
    all_lines.append(header)
    for p in prompts:
        r = generate_reply(p)
        print(f"USER: {p}\nTINYGPT: {r}\n")
        all_lines.append(f"USER: {p}\nTINYGPT: {r}\n")

(SFT_DIR / "test_outputs_seeds.txt").write_text("\n".join(all_lines))
print("Saved to:", SFT_DIR / "test_outputs_seeds.txt")


######## SEED 7 ########
USER: Hello, how are you?
TINYGPT: What is the role of a user's client's identity? A person's identity is the responsibility of a user's identity. It is also the

USER: What do you like to do in your free time?
TINYGPT: A Simple Data? The following is a good idea for the user. You can also use a simple data model. What are the most common

USER: Would you like to go for a walk?
TINYGPT: What is the best way to do with your own computer? The key to creating a computer is to use an electronic computer, which can be used

USER: I'm feeling sad today.
TINYGPT: "The term 'the term's' is used to describe the "the name of a person" or "the name of a person", or

USER: Can you help me with something?
TINYGPT: The Law of the Law The Law of the Law is a law of law. It is a law that does not mean that it has a law

USER: What did you do yesterday?
TINYGPT: The United States Department of Public Health. (2016). http://www.ncbi.gov/health/t_s_d_f_

USER: Do you like music?

In [ ]:
import torch
from pathlib import Path
from transformers import GPT2Config, GPT2LMHeadModel, AutoTokenizer

# 1. Load TinyGPT
CHECKPOINT_PATH = Path(
    "/content/drive/MyDrive/tiny-GPT-v2/"
    "sft_stage2_checkpoints/best_sft_model.pt"
)

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

config = GPT2Config.from_dict(checkpoint["config"])
config.use_cache = True
config.pad_token_id = tokenizer.eos_token_id

model = GPT2LMHeadModel(config)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

del checkpoint

print("✅ TinyGPT is ready!")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


# 2. Chat with TinyGPT
while True:
    prompt = input("\nYou: ").strip()

    if prompt.lower() in {"exit", "quit"}:
        print("TinyGPT: Goodbye! 👋")
        break

    if not prompt:
        continue

    formatted_prompt = f"User: {prompt}\nAssistant:"

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt"
    ).to(device)

    max_new_tokens = 60

    # Keep input + generated tokens within the model's context.
    max_prompt_length = model.config.n_positions - max_new_tokens
    inputs["input_ids"] = inputs["input_ids"][:, -max_prompt_length:]
    inputs["attention_mask"] = inputs["attention_mask"][:, -max_prompt_length:]

    with torch.no_grad():
        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device == "cuda")
        ):
            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                repetition_penalty=1.1,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id
            )

    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

    print("TinyGPT:", response or "(No response generated)")

✅ TinyGPT is ready!
Parameters: 51,082,752

You: Hello , I am Siddharth.
TinyGPT: Hi, you are welcome to be here.

You: Can you help me ? 
TinyGPT: I'd like to buy some new products.

You: Hey have you seen Sam around ? 
TinyGPT: Really? I haven't heard it was.

You: Did you watched that movie? 
TinyGPT: It was a big game.

You: Write a poem.
TinyGPT: You're a little tired!

You: Where is your girlfriend
TinyGPT: She is very sorry to be married.

You: but why ? 
TinyGPT: I'm sorry.

You: fuck you
TinyGPT: Yes, I'm afraid it's not too much.

You: Have you felt your dick is small ? 
TinyGPT: Yes, I have.

You: then your wife wouldn't be happy about that
TinyGPT: I'll go to the doctor for you.

You: but you have the problem
TinyGPT: I'm not sure. But I'm sorry, I don't know what the problem is.

You: you have rabbies
TinyGPT: yes, I think.

You: then you will have hydrophobia too
TinyGPT: well, I can't say that.

You: Have you listned about graphics ?
TinyGPT: Yes, I have.

You: What they

KeyboardInterrupt: Interrupted by user

In [ ]:
import torch
from pathlib import Path
from transformers import GPT2Config, GPT2LMHeadModel, AutoTokenizer

# Load the best fine-tuned TinyGPT
path = Path(
    "/content/drive/MyDrive/tiny-GPT-v2/"
    "sft_stage2_checkpoints/best_sft_model.pt"
)

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

ckpt = torch.load(path, map_location="cpu", weights_only=False)

config = GPT2Config.from_dict(ckpt["config"])
config.use_cache = True
config.pad_token_id = tokenizer.eos_token_id

model = GPT2LMHeadModel(config)
model.load_state_dict(ckpt["model_state_dict"])
model.to(device)
model.eval()

del ckpt

print("✅ Best SFT checkpoint loaded!")


def test_tinygpt(message):
    prompt = f"User: {message}\nAssistant:"

    inputs = tokenizer(
        prompt, return_tensors="pt"
    ).to(device)

    max_new_tokens = 40
    max_input = model.config.n_positions - max_new_tokens

    inputs["input_ids"] = inputs["input_ids"][:, -max_input:]
    inputs["attention_mask"] = inputs["attention_mask"][:, -max_input:]

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # Deterministic greedy decoding
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True
        )

    generated = output[0, inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(
        generated, skip_special_tokens=True
    ).strip()

    # Stop displaying if the model starts another conversation turn.
    answer = answer.split("\nUser:")[0].split("\nAssistant:")[0]

    print(f"\nYou: {message}")
    print(f"TinyGPT: {answer}")


prompts = [
    "Hello, how are you?",
    "What do you like to do in your free time?",
    "I'm feeling sad today.",
    "Can you explain what a GPU does?",
    "Would you like to go for a walk?",
    "Can you help me?",
]

for prompt in prompts:
    test_tinygpt(prompt)

✅ Best SFT checkpoint loaded!

You: Hello, how are you?
TinyGPT: I'm going to the airport.

You: What do you like to do in your free time?
TinyGPT: I like to do some things.

You: I'm feeling sad today.
TinyGPT: I'm not sure. I'm not sure.

You: Can you explain what a GPU does?
TinyGPT: I'm not sure. I'm not sure.

You: Would you like to go for a walk?
TinyGPT: Sure. I'd like to go to the gym.

You: Can you help me?
TinyGPT: Yes, I'd like to buy a new one.
